In [2]:

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'openpyxl', '-q'])
print("ok")


ok


In [5]:

import re, ast
from pathlib import Path
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ─── Leggi mazzi.ts ────────────────────────────────────────────────
src = Path('/workspace/linea_rossa_tracker/src/data/mazzi.ts').read_text()

# ─── Regex per estrarre oggetti carta ─────────────────────────────
# Pattern: { card_id:'...', ..., effects:{...} }
card_pattern = re.compile(
    r"\{\s*card_id:\s*'([^']+)',\s*card_name:\s*'([^']+)',\s*faction:\s*'([^']+)',\s*card_type:\s*'([^']+)',\s*op_points:\s*(\d+),\s*deck_type:\s*'([^']+)',\s*description:\s*'([^']*)'(?:,\s*unlocks_special:\s*true)?,\s*effects:\s*(\{[^}]+\})\s*\}",
    re.DOTALL
)

# ─── Parser effetti: da TS lambda a delta numerico (default 5) ──
DEFAULTS = {
    'nucleare': 5, 'sanzioni': 10, 'opinione': 0, 'defcon': 7,
    'risorse': 5, 'stabilita': 5,
    'risorse_iran': 5, 'risorse_coalizione': 5, 'risorse_russia': 5,
    'risorse_cina': 5, 'risorse_europa': 5,
    'stabilita_iran': 5, 'stabilita_coalizione': 5, 'stabilita_russia': 5,
    'stabilita_cina': 5, 'stabilita_europa': 5,
}

def parse_effects(effects_str):
    """Ritorna dict {chiave: delta_numerico} usando i valori default."""
    # Estrai coppie chiave: (v)=>espressione
    pair_re = re.compile(
        r'([\w_]+)\s*:\s*\(v\)\s*=>\s*([^\n,}]+)',
        re.DOTALL
    )
    result = {}
    for key, expr in pair_re.findall(effects_str):
        key = key.strip()
        expr = expr.strip().rstrip(',')
        v = DEFAULTS.get(key, 5)
        try:
            # Sostituisci sintassi ternaria TS in Python
            # es. v<=7?2:v<=12?1:3 → (2 if v<=7 else (1 if v<=12 else 3))
            py_expr = expr
            # Converti ternari multipli: "a?b:c" → "(b if a else c)"
            def conv_ternary(s):
                # Tratta solo il primo livello
                m = re.match(r'^(.+?)\?(.+?):(.+)$', s)
                if not m:
                    return s
                cond = m.group(1).strip()
                then = m.group(2).strip()
                els  = m.group(3).strip()
                then = conv_ternary(then)
                els  = conv_ternary(els)
                return f'({then} if {cond} else {els})'
            py_expr = conv_ternary(py_expr)
            delta = eval(py_expr, {'v': v, '__builtins__': {}})
            result[key] = int(delta) if delta == int(delta) else float(delta)
        except Exception as e:
            result[key] = '?'
    return result

# ─── Estrai tutte le carte ────────────────────────────────────────
cards = []
for m in card_pattern.finditer(src):
    cid, name, faction, ctype, op, deck, desc, eff_str = m.groups()
    effs = parse_effects(eff_str)
    cards.append({
        'card_id': cid, 'card_name': name, 'faction': faction,
        'card_type': ctype, 'op_points': int(op), 'deck_type': deck,
        'description': desc, 'effects_raw': eff_str.replace('\n','').strip(),
        'effects': effs,
    })

print(f"Carte estratte: {len(cards)}")

# Conta carte con effetti multi-fazione
MULTI_KEYS = [
    'risorse_iran','risorse_coalizione','risorse_russia','risorse_cina','risorse_europa',
    'stabilita_iran','stabilita_coalizione','stabilita_russia','stabilita_cina','stabilita_europa',
]
multi_count = sum(1 for c in cards if any(k in c['effects'] for k in MULTI_KEYS))
print(f"Carte con effetti multi-fazione espliciti: {multi_count}")


Carte estratte: 333
Carte con effetti multi-fazione espliciti: 0


In [8]:

from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FACTIONS_ORDER = ['Iran', 'Coalizione', 'Russia', 'Cina', 'Europa']

def compute_faction_deltas(card):
    """
    Per ogni fazione calcola delta risorse e stabilita:
    - Se la carta è di quella fazione → prende da 'risorse'/'stabilita' (effetto su sé stessa)
    - Se faction='Neutrale' → 'risorse'/'stabilita' si applica a TUTTE le fazioni
    - Poi somma l'effetto esplicito risorse_<faz> / stabilita_<faz>
    """
    faction = card['faction']
    effs    = card['effects']
    
    def val(k):
        v = effs.get(k, 0)
        return 0 if v == '?' else (v or 0)
    
    out = {}
    for f in FACTIONS_ORDER:
        fk = f.lower()
        ris = 0
        sta = 0
        if faction == f:
            ris += val('risorse')
            sta += val('stabilita')
        elif faction == 'Neutrale':
            ris += val('risorse')
            sta += val('stabilita')
        # Effetto esplicito su fazione specifica
        ris += val(f'risorse_{fk}')
        sta += val(f'stabilita_{fk}')
        out[f'ris_{fk}'] = ris
        out[f'sta_{fk}'] = sta
    return out

# Costruisci righe Excel
rows = []
for c in cards:
    effs = c['effects']
    def g(k): return effs.get(k, 0) or 0
    fd = compute_faction_deltas(c)
    
    # Stringa effetti leggibile
    eff_parts = []
    for k, v in sorted(effs.items()):
        if v and v != 0 and v != '?':
            sign = '+' if v > 0 else ''
            eff_parts.append(f"{k}:{sign}{v}")
    eff_str = '  '.join(eff_parts)
    
    row = {
        'A_card_id':    c['card_id'],
        'B_card_name':  c['card_name'],
        'C_faction':    c['faction'],
        'D_card_type':  c['card_type'],
        'E_op_points':  c['op_points'],
        'F_deck_type':  c['deck_type'],
        'G_description': c['description'],
        # Globali
        'I_nucleare':   g('nucleare'),
        'J_sanzioni':   g('sanzioni'),
        'K_opinione':   g('opinione'),
        'L_defcon':     g('defcon'),
        # Risorse per fazione
        'M_ris_iran':       fd['ris_iran'],
        'N_ris_coalizione': fd['ris_coalizione'],
        'O_ris_russia':     fd['ris_russia'],
        'P_ris_cina':       fd['ris_cina'],
        'Q_ris_europa':     fd['ris_europa'],
        # Stabilità per fazione
        'R_sta_iran':       fd['sta_iran'],
        'S_sta_coalizione': fd['sta_coalizione'],
        'T_sta_russia':     fd['sta_russia'],
        'U_sta_cina':       fd['sta_cina'],
        'V_sta_europa':     fd['sta_europa'],
        # Stringa effetti
        'W_effects':    eff_str,
    }
    rows.append(row)

print(f"Righe dati: {len(rows)}")
print("Esempio prima riga:", {k: v for k, v in list(rows[0].items())[:8]})


Righe dati: 333
Esempio prima riga: {'A_card_id': 'C025', 'B_card_name': 'Arricchimento Uranio 60%', 'C_faction': 'Iran', 'D_card_type': 'Militare', 'E_op_points': 4, 'F_deck_type': 'base', 'G_description': 'Accelerazione del programma nucleare', 'I_nucleare': 2}


In [11]:

import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Carte Linea Rossa"

# ─── Header ────────────────────────────────────────────────────────
headers = [
    ('A', 'ID'),
    ('B', 'Nome Carta'),
    ('C', 'Fazione'),
    ('D', 'Tipo'),
    ('E', 'OP'),
    ('F', 'Mazzo'),
    ('G', 'Descrizione'),
    ('H', ''),
    ('I', '☢️ Nucleare'),
    ('J', '🔒 Sanzioni'),
    ('K', '📣 Opinione'),
    ('L', '🚨 DEFCON'),
    ('M', '💵🇮🇷 Iran'),
    ('N', '💵🇺🇸 Coal.'),
    ('O', '💵🇷🇺 Russia'),
    ('P', '💵🇨🇳 Cina'),
    ('Q', '💵🇪🇺 Europa'),
    ('R', '⚖️🇮🇷 Iran'),
    ('S', '⚖️🇺🇸 Coal.'),
    ('T', '⚖️🇷🇺 Russia'),
    ('U', '⚖️🇨🇳 Cina'),
    ('V', '⚖️🇪🇺 Europa'),
    ('W', 'Effetti (stringa)'),
]

# Colori header per sezione
HDR_COLORS = {
    'A': 'D6E4F0', 'B': 'D6E4F0', 'C': 'D6E4F0', 'D': 'D6E4F0',
    'E': 'D6E4F0', 'F': 'D6E4F0', 'G': 'D6E4F0', 'H': 'FFFFFF',
    'I': 'FFE0B2', 'J': 'FFE0B2', 'K': 'FFE0B2', 'L': 'FFE0B2',
    'M': 'E8F5E9', 'N': 'E8F5E9', 'O': 'E8F5E9', 'P': 'E8F5E9', 'Q': 'E8F5E9',
    'R': 'F3E5F5', 'S': 'F3E5F5', 'T': 'F3E5F5', 'U': 'F3E5F5', 'V': 'F3E5F5',
    'W': 'FFFDE7',
}

FACTION_COLORS = {
    'Iran':       'FFD0D0',
    'Coalizione': 'D0E4FF',
    'Russia':     'FFE4D0',
    'Cina':       'FFD0D0',
    'Europa':     'D0FFD0',
    'Neutrale':   'F0F0F0',
}

# ─── Colonne larghezze ─────────────────────────────────────────────
col_widths = {
    1: 10, 2: 30, 3: 14, 4: 13, 5: 5, 6: 10, 7: 35, 8: 3,
    9: 10, 10: 10, 11: 10, 12: 10,
    13: 9, 14: 9, 15: 9, 16: 9, 17: 9,
    18: 9, 19: 9, 20: 9, 21: 9, 22: 9,
    23: 60
}
for col_idx, width in col_widths.items():
    ws.column_dimensions[get_column_letter(col_idx)].width = width

# ─── Scrivi header (riga 1) ────────────────────────────────────────
for col_idx, (col_letter, hdr) in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx, value=hdr)
    bg = HDR_COLORS.get(col_letter, 'FFFFFF')
    cell.fill = PatternFill('solid', fgColor=bg)
    cell.font = Font(bold=True, size=10)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

ws.row_dimensions[1].height = 32
ws.freeze_panes = 'A2'

# ─── Scrivi dati ──────────────────────────────────────────────────
def fmt_delta(v):
    """Formatta delta: 0 → '', positivi con +, negativi con -"""
    if v == 0 or v == '?': return ''
    return f'+{v}' if v > 0 else str(v)

for row_idx, row in enumerate(rows, start=2):
    vals = list(row.values())
    faction = vals[2]
    row_bg = FACTION_COLORS.get(faction, 'FFFFFF')
    
    for col_idx, val in enumerate(vals, start=1):
        cell = ws.cell(row=row_idx, column=col_idx)
        # Colonne delta numeriche: formatta come stringa con segno
        if col_idx in range(9, 23):
            cell.value = fmt_delta(val)
            cell.alignment = Alignment(horizontal='center', vertical='center')
            # Colora: verde se positivo, rosso se negativo
            if val and val != 0:
                is_positive = val > 0
                # Per nucleare e sanzioni il positivo non è necessariamente buono
                # Ma per semplicità, verde=positivo, rosso=negativo
                if is_positive:
                    cell.fill = PatternFill('solid', fgColor='C8E6C9')
                    cell.font = Font(color='1B5E20', bold=True)
                else:
                    cell.fill = PatternFill('solid', fgColor='FFCDD2')
                    cell.font = Font(color='B71C1C', bold=True)
            else:
                cell.fill = PatternFill('solid', fgColor=row_bg)
        elif col_idx == 23:  # stringa effetti
            cell.value = val
            cell.alignment = Alignment(horizontal='left', vertical='center', wrap_text=False)
            cell.fill = PatternFill('solid', fgColor='FFFFF0')
        else:
            cell.value = val
            cell.alignment = Alignment(horizontal='left', vertical='center')
            cell.fill = PatternFill('solid', fgColor=row_bg)
    
    # Altezza riga
    ws.row_dimensions[row_idx].height = 18

# ─── Salva ────────────────────────────────────────────────────────
out_path = '/workspace/linea_rossa_carte_completo.xlsx'
wb.save(out_path)
print(f"Salvato: {out_path}")
print(f"Totale righe dati: {len(rows)}")

# Conta per fazione
from collections import Counter
faction_counts = Counter(r['C_faction'] for r in rows)
for fac, cnt in sorted(faction_counts.items()):
    multi = sum(1 for r in rows if r['C_faction'] == fac and any(
        (r.get(k, 0) or 0) != 0 for k in [
            'M_ris_iran','N_ris_coalizione','O_ris_russia','P_ris_cina','Q_ris_europa',
            'R_sta_iran','S_sta_coalizione','T_sta_russia','U_sta_cina','V_sta_europa'
        ]
    ))
    print(f"  {fac}: {cnt} carte, {multi} con qualche delta fazione ≠ 0")


TypeError: '>' not supported between instances of 'str' and 'int'